# R-GFM — Full Reproduction (single notebook)

One notebook, run top to bottom, that reproduces every result the **official R-GFM code** (`USTC-DataDarknessLab/R-GFM`) can actually produce:

- **Table 1** — 1-shot node classification, 8 datasets, leave-one-dataset-out
- **Table 2 / Table 3** — 3-shot / 5-shot node classification, same 8 datasets
- **Table 9** (Table 5 is a subset of it) — link prediction (AUC-ROC), 7 datasets
- A final **comparison summary** against the paper's reported numbers

This merges what were previously five separate notebooks (`00_setup` … `04_results_summary`) into one, so there's a single running kernel and no cross-notebook state-tracking to worry about. The separate notebooks still exist side by side if you'd rather run smaller pieces independently or in different Colab sessions.

## What this notebook does **not** reproduce, and why

The paper reports more than this, but the rest genuinely isn't reproducible from what the authors released in this repo (checked against the `main` branch):

| Paper result | Why it's out of scope here |
|---|---|
| Figure 4 ablations (Euclidean-only GoG, chain-topology edges, single-manifold experts, w/o GoG pooling, Stage-1-only) | None of these variants are exposed as CLI flags in `parser/parser_node_level.py` — they'd require editing the model/trainer source, not just calling `main.py` differently. |
| Figure 6a (`Bedge` sensitivity) | The edge budget isn't a CLI argument; it's derived internally from `k_max_hop`, not settable directly. |
| Figure 5 (accuracy vs hop number K) and Figure 6b (temperature `tau`) | *Could* be swept via the existing `--k_max_hop` / `--tau` flags — genuinely reproducible with more runs, just not included here to keep this notebook's scope to the paper's headline tables. Ask if you want a section added for these. |
| Table 4 (large-scale LM-feature setting: ogbn-Arxiv/ArXiv_2023/Reddit → Cora/Ele-Computers/Books-History/Instagram) | This repo's `utils/data/loader.py` only supports `{cora, citeseer, pubmed, cornell, texas, wisconsin, chameleon, squirrel, computers, photo, penn94, ogbl-collab}` — none of the large-scale-setting datasets have a loader here. |
| Tables 10/11 (graph classification on ChEMBL → HIV) | No ChEMBL/HIV loader or graph-classification training path exists in this repo at all. |

## Known repo quirks (already worked around below)

- The repo's own README references a `requirements.txt` that doesn't exist in the repo — dependencies below are inferred from actual source imports.
- `graph_aug/cuda_backend.py` is dead code (JIT-compiles from source files that don't exist); the real build path is `setup.py build_ext --inplace`, which is internally consistent and is what's used here.
- Chameleon/Squirrel are excluded as **evaluation targets** below (matching the paper's own Table 1 methodology, due to documented duplicate-node/train-test-leakage issues — Platonov et al. 2023), though they remain available as automatic pretraining sources for the other 8 datasets (the leave-one-dataset-out logic is hardcoded in `trainers/node2graph_trainer.py`, not a CLI flag).

**Before running:** Runtime -> Change runtime type -> GPU. This notebook trains 8 datasets x 3 shot settings + 7 link-prediction datasets = 31 full training runs; on free-tier Colab you will very likely hit a GPU-quota wall partway through. That's fine — every loop below skips datasets whose result log already exists, so you can stop, come back after quota resets, re-run this same notebook top to bottom, and it'll pick up where it left off (only Section 0 — the repo clone / pip installs / CUDA build — needs to fully redo on a fresh runtime, since that lives on ephemeral VM disk, not Drive).

## Section 0 — Setup

In [1]:
# GPU check.
!nvidia-smi | head -n 15

Mon Aug 24 06:27:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone the official repository.
import os
REPO_DIR = '/content/R-GFM'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/USTC-DataDarknessLab/R-GFM.git $REPO_DIR
else:
    print(f'{REPO_DIR} already exists — pulling latest')
    !git -C $REPO_DIR pull --ff-only

Cloning into '/content/R-GFM'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 61 (delta 7), reused 57 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 38.39 KiB | 9.60 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [3]:
!ls -la $REPO_DIR
!nvcc --version

total 52
drwxr-xr-x 8 root root 4096 Aug 24 06:27 .
drwxr-xr-x 1 root root 4096 Aug 24 06:27 ..
drwxr-xr-x 8 root root 4096 Aug 24 06:27 .git
drwxr-xr-x 4 root root 4096 Aug 24 06:27 graph_aug
-rw-r--r-- 1 root root 1121 Aug 24 06:27 LICENSE
-rw-r--r-- 1 root root  387 Aug 24 06:27 main-link.py
-rw-r--r-- 1 root root  736 Aug 24 06:27 main.py
drwxr-xr-x 5 root root 4096 Aug 24 06:27 models
drwxr-xr-x 2 root root 4096 Aug 24 06:27 parser
-rw-r--r-- 1 root root 4383 Aug 24 06:27 readme.md
drwxr-xr-x 2 root root 4096 Aug 24 06:27 trainers
drwxr-xr-x 6 root root 4096 Aug 24 06:27 utils
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [4]:
# torch pinned per the repo's stated requirement; detect the exact build string
# so the PyG wheel index (next cell) matches it exactly.
!pip install --quiet torch==2.8.0

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
print('torch', torch.__version__, '-> wheel tag', TORCH_VER, CUDA_TAG, '| cuda available:', torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.8.0 which is incompatible.
torch 2.8.0+cu128 -> wheel tag 2.8.0 cu128 | cuda available: True


In [5]:
# torch_geometric + torch_scatter/torch_sparse matched to the detected build.
wheel_url = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'
print('PyG wheel index:', wheel_url)
!pip install --quiet torch_geometric
os.system(f'pip install --quiet torch_scatter torch_sparse -f {wheel_url}')

PyG wheel index: https://data.pyg.org/whl/torch-2.8.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.9 MB/s eta 0:00:00


0

In [6]:
# Remaining dependencies (no requirements.txt exists upstream — see markdown above).
# geoopt==0.5.0 would hit the same scipy.optimize.linesearch removal issue documented in
# baseline_reproduction/00_setup.ipynb; taking current geoopt avoids that entirely here
# since R-GFM's repo doesn't pin a geoopt version either.
!pip install --quiet geoopt ogb scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 7.2 MB/s eta 0:00:00


In [7]:
# Build the custom CUDA graph-augmentation extension — required, main.py crashes on
# import otherwise (utils/data/augmentation.py imports graph_aug.graph_aug_cuda).
%cd $REPO_DIR/graph_aug
!python setup.py build_ext --inplace
%cd $REPO_DIR

/content/R-GFM/graph_aug
running build_ext
W0824 06:29:49.745000 1306 torch/utils/cpp_extension.py:615] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
W0824 06:29:49.829000 1306 torch/utils/cpp_extension.py:517] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.8
building 'graph_aug_cuda' extension
creating build/temp.linux-x86_64-cpython-313/cpp
creating build/temp.linux-x86_64-cpython-313/cuda_kernels
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -fPIC -I/usr/local/lib/python3.13/dist-packages/torch/include -I/usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -I/usr/local/cuda/include -I/usr/include/python3.13 -c cpp/graph_aug.cpp -o build/temp.linux-x86_64-cpython-313/cpp/graph_aug.o -O3 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_

In [9]:
# Mount Drive for persistent datasets/checkpoints/results across session resets.
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/R-GFM'
DATA_DIR = f'{BASE}/datasets'
CKPT_DIR = f'{BASE}/checkpoints'
RESULTS_DIR = f'{BASE}/results'
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
print('Drive workspace:', BASE)

Mounted at /content/drive
Drive workspace: /content/drive/MyDrive/R-GFM


In [10]:
# Sanity check — imports, versions, and the built extension all resolve.
import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import torch_geometric, geoopt, ogb
print('python', sys.version.split()[0])
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
print('pyg', torch_geometric.__version__)
print('geoopt', geoopt.__version__)
print('ogb', ogb.__version__)

import graph_aug.graph_aug_cuda as graph_aug_cuda
print('graph_aug_cuda loaded OK:', [n for n in dir(graph_aug_cuda) if not n.startswith('_')])

python 3.13.15
torch 2.8.0+cu128 cuda_available True
pyg 2.8.0.post1
geoopt 0.5.1
ogb 1.3.6
graph_aug_cuda loaded OK: ['drop_nodes_batch_forward', 'mask_nodes_batch_forward', 'permute_edges_batch_forward']


## Section 1 — Node Classification (Tables 1, 2, 3)

For `--dataset X`, `main.py` automatically pretrains on all *other* datasets in the 10-dataset pool and evaluates transfer on `X` (leave-one-dataset-out — hardcoded in the trainer, confirmed by reading `trainers/node2graph_trainer.py`). `--shots N` controls k in the k-shot fine-tuning split.

Each shot setting loops over all 8 datasets and **skips any dataset whose log already exists** — safe to stop and re-run this notebook after a GPU-quota interruption without redoing finished work. Delete a log file to force that one dataset to re-run.

In [11]:
NC_DATASETS = ['wisconsin', 'texas', 'cornell', 'citeseer', 'cora', 'pubmed', 'computers', 'photo']

def run_node_classification(shots, tag):
    log_dir = f'{RESULTS_DIR}/nc_{tag}'
    os.makedirs(log_dir, exist_ok=True)
    for ds in NC_DATASETS:
        log = f'{log_dir}/{ds}.log'
        if os.path.exists(log):
            print(f'--- {tag} / {ds}: log already exists, skipping (delete it to force a re-run) ---')
            continue
        print(f'=== {tag} ({shots}-shot): {ds} ===')
        !python main.py --dataset {ds} --epochs 150 --shots {shots} --device 0 \
            --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_{tag}_{ds} \
            2>&1 | tee $log
    return log_dir

In [12]:
# 1-shot (Table 1)
NC_1SHOT_DIR = run_node_classification(shots=1, tag='1shot')

=== 1shot (1-shot): wisconsin ===
Processing...
Done!
Namespace(device=0, dataset_dir='/content/drive/MyDrive/R-GFM/datasets', dataset='wisconsin', shots=1, k_max_hop=6, proj_dim=64, tau=0.2, encoder_epochs=100, epochs=150, stage1_sim_agg=True, sim_agg_alpha=0.1, load_balance_weight=0.01, topm_start=3, topm_min=1, topm_lb_thresh=0.05, checkpoint_dir='/content/drive/MyDrive/R-GFM/checkpoints', checkpoint_prefix='node2graph_1shot_wisconsin', seed=42)
Data for texas not found. Downloading and processing...
texas saved to /content/drive/MyDrive/R-GFM/datasets/texas
Generating node-based subgraphs: 100%|██████████| 183/183 [00:00<00:00, 214.42node/s]
Processing...
Done!
Data for cornell not found. Downloading and processing...
cornell saved to /content/drive/MyDrive/R-GFM/datasets/cornell
Generating node-based subgraphs: 100%|██████████| 183/183 [00:00<00:00, 242.67node/s]
Processing...
Done!
Data for cora not found. Downloading and processing...
cora saved to /content/drive/MyDrive/R-GFM/d

In [13]:
# 3-shot (Table 2)
NC_3SHOT_DIR = run_node_classification(shots=3, tag='3shot')

=== 3shot (3-shot): wisconsin ===
Namespace(device=0, dataset_dir='/content/drive/MyDrive/R-GFM/datasets', dataset='wisconsin', shots=3, k_max_hop=6, proj_dim=64, tau=0.2, encoder_epochs=100, epochs=150, stage1_sim_agg=True, sim_agg_alpha=0.1, load_balance_weight=0.01, topm_start=3, topm_min=1, topm_lb_thresh=0.05, checkpoint_dir='/content/drive/MyDrive/R-GFM/checkpoints', checkpoint_prefix='node2graph_3shot_wisconsin', seed=42)
Found preprocessed data for texas, loading...
Traceback (most recent call last):
  File "/content/R-GFM/main.py", line 27, in <module>
    main()
    ~~~~^^
  File "/content/R-GFM/main.py", line 14, in main
    trainer = Node2GraphTrainer(args)
  File "/content/R-GFM/trainers/node2graph_trainer.py", line 52, in __init__
    self._prepare_data()
    ~~~~~~~~~~~~~~~~~~^^
  File "/content/R-GFM/trainers/node2graph_trainer.py", line 102, in _prepare_data
    feature, y, graphs_list, graphs_index_list = load_single_dataset(ds_name)
                                  

In [14]:
# 5-shot (Table 3)
NC_5SHOT_DIR = run_node_classification(shots=5, tag='5shot')

=== 5shot (5-shot): wisconsin ===
Namespace(device=0, dataset_dir='/content/drive/MyDrive/R-GFM/datasets', dataset='wisconsin', shots=5, k_max_hop=6, proj_dim=64, tau=0.2, encoder_epochs=100, epochs=150, stage1_sim_agg=True, sim_agg_alpha=0.1, load_balance_weight=0.01, topm_start=3, topm_min=1, topm_lb_thresh=0.05, checkpoint_dir='/content/drive/MyDrive/R-GFM/checkpoints', checkpoint_prefix='node2graph_5shot_wisconsin', seed=42)
Found preprocessed data for texas, loading...
Traceback (most recent call last):
  File "/content/R-GFM/main.py", line 27, in <module>
    main()
    ~~~~^^
  File "/content/R-GFM/main.py", line 14, in main
    trainer = Node2GraphTrainer(args)
  File "/content/R-GFM/trainers/node2graph_trainer.py", line 52, in __init__
    self._prepare_data()
    ~~~~~~~~~~~~~~~~~~^^
  File "/content/R-GFM/trainers/node2graph_trainer.py", line 102, in _prepare_data
    feature, y, graphs_list, graphs_index_list = load_single_dataset(ds_name)
                                  

In [15]:
# Quick summary across all three shot settings.
import re

def summarize_nc(log_dir, datasets):
    for ds in datasets:
        log = f'{log_dir}/{ds}.log'
        if not os.path.exists(log):
            print(f'  {ds:12s}  (not run yet)')
            continue
        txt = open(log).read()
        m = re.findall(r'Test Accuracy:\s*([0-9.]+)\s*\+/-\s*([0-9.]+)', txt)
        if m:
            acc, std = m[-1]
            print(f'  {ds:12s}  {float(acc)*100:.2f} +/- {float(std)*100:.2f}')
        else:
            print(f'  {ds:12s}  (no result line found — check log)')

print('--- 1-shot ---'); summarize_nc(NC_1SHOT_DIR, NC_DATASETS)
print('\n--- 3-shot ---'); summarize_nc(NC_3SHOT_DIR, NC_DATASETS)
print('\n--- 5-shot ---'); summarize_nc(NC_5SHOT_DIR, NC_DATASETS)

--- 1-shot ---
  wisconsin     (no result line found — check log)
  texas         (no result line found — check log)
  cornell       (no result line found — check log)
  citeseer      (no result line found — check log)
  cora          (no result line found — check log)
  pubmed        (no result line found — check log)
  computers     (no result line found — check log)
  photo         (no result line found — check log)

--- 3-shot ---
  wisconsin     (no result line found — check log)
  texas         (no result line found — check log)
  cornell       (no result line found — check log)
  citeseer      (no result line found — check log)
  cora          (no result line found — check log)
  pubmed        (no result line found — check log)
  computers     (no result line found — check log)
  photo         (no result line found — check log)

--- 5-shot ---
  wisconsin     (no result line found — check log)
  texas         (no result line found — check log)
  cornell       (no result line fou

**Target numbers (paper, R-GFM row, accuracy %).**

1-shot (Table 1):

| Wisconsin | Cornell | Citeseer | Cora | Pubmed | Computers | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 35.41 ± 7.29 | 36.71 ± 9.92 | 57.54 ± 9.49 | 49.50 ± 3.97 | 49.80 ± 5.38 | 52.30 ± 3.33 | 61.08 ± 5.26 | 32.36 ± 12.10 |

3-shot (Table 2):

| Wisconsin | Cornell | Citeseer | Cora | Pubmed | Computers | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 43.10 ± 4.03 | 45.39 ± 3.09 | 73.98 ± 1.42 | 59.26 ± 5.04 | 59.19 ± 3.01 | 56.02 ± 3.90 | 71.50 ± 2.80 | 44.18 ± 7.02 |

5-shot (Table 3):

| Wisconsin | Cornell | Citeseer | Cora | Pubmed | Computers | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 47.75 ± 4.97 | 47.97 ± 2.82 | 74.59 ± 1.87 | 63.77 ± 2.47 | 63.39 ± 1.85 | 59.55 ± 4.17 | 73.62 ± 1.15 | 51.64 ± 6.31 |

## Section 2 — Link Prediction (Table 9 / Table 5)

`main-link.py` (Edge2Graph) trains its own encoder + MoE stage per dataset and prints:

```
[Test] Acc <acc> | AUC <auc> | Hits@<k> <hits>
```

Same skip-if-log-exists behavior as Section 1.

In [ ]:
LP_DATASETS = ['wisconsin', 'cornell', 'citeseer', 'pubmed', 'cora', 'photo', 'texas']
LP_LOG_DIR = f'{RESULTS_DIR}/lp'
os.makedirs(LP_LOG_DIR, exist_ok=True)

for ds in LP_DATASETS:
    log = f'{LP_LOG_DIR}/{ds}.log'
    if os.path.exists(log):
        print(f'--- {ds}: log already exists, skipping (delete it to force a re-run) ---')
        continue
    print(f'=== link prediction: {ds} ===')
    !python main-link.py --dataset {ds} --epochs 200 --device 0 \
        --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix edge2graph_{ds} \
        2>&1 | tee $log

=== link prediction: wisconsin ===
Namespace(device=0, dataset_dir='/content/drive/MyDrive/R-GFM/datasets', dataset='wisconsin', k_max_hop=5, shots=50, neg_ratio=1, encoder_lr=0.005, moe_lr=0.005, classifier_lr=0.005, riemannian_lr=0.001, weight_decay=2e-06, encoder_epochs=50, epochs=200, stage1_sim_agg=False, sim_agg_alpha=0.1, data_sample_ratio=1.0, edge_sample_ratio=1.0, load_balance_weight=0.01, topm_start=3, topm_min=1, topm_lb_thresh=0.05, checkpoint_dir='/content/drive/MyDrive/R-GFM/checkpoints', checkpoint_prefix='edge2graph_wisconsin', seed=42)
Found preprocessed data for wisconsin, loading...
[train] Cached subgraphs saved: /content/drive/MyDrive/R-GFM/datasets/processed_data/wisconsin/link_khop_k5_split_train_seed42_ratio1.0_edges1.0.pt
[val] Cached subgraphs saved: /content/drive/MyDrive/R-GFM/datasets/processed_data/wisconsin/link_khop_k5_split_val_seed42_ratio1.0_edges1.0.pt
[test] Cached subgraphs saved: /content/drive/MyDrive/R-GFM/datasets/processed_data/wisconsin/link

In [ ]:
# Quick summary.
for ds in LP_DATASETS:
    log = f'{LP_LOG_DIR}/{ds}.log'
    if not os.path.exists(log):
        print(f'{ds:12s}  (not run yet)')
        continue
    txt = open(log).read()
    m = re.findall(r'\[Test\]\s*Acc\s*([0-9.]+)\s*\|\s*AUC\s*([0-9.]+)\s*\|\s*Hits@(\d+)\s*([0-9.]+)', txt)
    if m:
        acc, auc, k, hits = m[-1]
        print(f'{ds:12s}  Acc {float(acc)*100:.2f}  AUC {float(auc)*100:.2f}  Hits@{k} {float(hits)*100:.2f}')
    else:
        print(f'{ds:12s}  (no result line found — check log)')

**Target numbers (paper's Table 9, R-GFM row, AUC-ROC %).**

| Wisconsin | Cornell | Citeseer | Pubmed | Cora | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|
| 84.15 ± 0.71 | 85.90 ± 0.69 | 90.88 ± 0.70 | 88.62 ± 0.41 | 89.27 ± 0.64 | 81.53 ± 0.83 | 87.94 ± 0.96 |

## Section 3 — Results summary (paper vs. ours)

In [ ]:
import csv

PAPER = {
    'nc_1shot': {'wisconsin': 35.41, 'texas': 32.36, 'cornell': 36.71, 'citeseer': 57.54,
                 'cora': 49.50, 'pubmed': 49.80, 'computers': 52.30, 'photo': 61.08},
    'nc_3shot': {'wisconsin': 43.10, 'texas': 44.18, 'cornell': 45.39, 'citeseer': 73.98,
                 'cora': 59.26, 'pubmed': 59.19, 'computers': 56.02, 'photo': 71.50},
    'nc_5shot': {'wisconsin': 47.75, 'texas': 51.64, 'cornell': 47.97, 'citeseer': 74.59,
                 'cora': 63.77, 'pubmed': 63.39, 'computers': 59.55, 'photo': 73.62},
    'lp': {'wisconsin': 84.15, 'cornell': 85.90, 'citeseer': 90.88, 'pubmed': 88.62,
           'cora': 89.27, 'photo': 81.53, 'texas': 87.94},
}

def parse_nc_log(path):
    if not os.path.exists(path):
        return None
    txt = open(path).read()
    m = re.findall(r'Test Accuracy:\s*([0-9.]+)\s*\+/-\s*([0-9.]+)', txt)
    return float(m[-1][0]) * 100 if m else None

def parse_lp_log(path):
    if not os.path.exists(path):
        return None
    txt = open(path).read()
    m = re.findall(r'\[Test\]\s*Acc\s*[0-9.]+\s*\|\s*AUC\s*([0-9.]+)', txt)
    return float(m[-1]) * 100 if m else None

rows = []
for ds, paper_val in PAPER['nc_1shot'].items():
    rows.append({'task': 'NC-1shot', 'dataset': ds, 'ours': parse_nc_log(f'{NC_1SHOT_DIR}/{ds}.log'), 'paper': paper_val})
for ds, paper_val in PAPER['nc_3shot'].items():
    rows.append({'task': 'NC-3shot', 'dataset': ds, 'ours': parse_nc_log(f'{NC_3SHOT_DIR}/{ds}.log'), 'paper': paper_val})
for ds, paper_val in PAPER['nc_5shot'].items():
    rows.append({'task': 'NC-5shot', 'dataset': ds, 'ours': parse_nc_log(f'{NC_5SHOT_DIR}/{ds}.log'), 'paper': paper_val})
for ds, paper_val in PAPER['lp'].items():
    rows.append({'task': 'LP-AUC', 'dataset': ds, 'ours': parse_lp_log(f'{LP_LOG_DIR}/{ds}.log'), 'paper': paper_val})

for r in rows:
    r['delta'] = round(r['ours'] - r['paper'], 2) if r['ours'] is not None else None

out_csv = f'{RESULTS_DIR}/rgfm_comparison.csv'
with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['task', 'dataset', 'ours', 'paper', 'delta'])
    w.writeheader()
    for r in rows:
        w.writerow(r)
print('Wrote', out_csv)

print()
print('| Task | Dataset | Ours | Paper | Delta |')
print('|------|---------|-----:|------:|------:|')
for r in rows:
    ours = f"{r['ours']:.2f}" if r['ours'] is not None else 'n/a'
    delta = f"{r['delta']:+.2f}" if r['delta'] is not None else 'n/a'
    print(f"| {r['task']} | {r['dataset']} | {ours} | {r['paper']:.2f} | {delta} |")

**Done.** `rgfm_comparison.csv` on Drive has the full table. Any `n/a` rows mean that run hasn't completed yet — re-run this notebook (Sections 1/2 skip completed datasets automatically) and re-run this summary cell.